In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:90% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:20pt;}
.inner_cell{font-size:20pt;}
div.text_cell_render pre code {font-size:20pt; line-height:30px;}
div.output {font-size:20pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:20pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:20pt;padding:5px;}
table.dataframe{font-size:20px;}
</style>
"""))

<font color="red" size="6"><b>ch14. 웹 데이터 수집</b></font>

# 1절. BeautifulSoup과 parser
    (정적 웹크롤링, 공공api사용)
    
`pip install bs4` 아나콘다를 설치하면 자동 설치되는 패키지에 포함
- 공식 사이트 : https://www.crummy.com/software/BeautifulSoup/
- documentation : https://www.crummy.com/software/BeautifulSoup/bs4/doc/

In [2]:
import requests # HTTP요청 처리하는 lib
# file:// == c:
# http://www.
from requests_file import FileAdapter

In [12]:
# 로컬에 있는 파일을 웹요청하듯이 읽어오는 작업
s = requests.Session()
s.mount("file://", FileAdapter()) # file://로 시작하는 url을 어댑터가 처리
response = s.get("file:///ai/lecNote/01_python/data/ch14_sample.html") # c:/ai/lecNote/01_python/data/ch14_sample.html
response

<Response [200]>

In [13]:
if response:
    print('해당 url에 접근함')
else:
    print('해당 url에 거부됨')

해당 url에 접근함


In [14]:
response.status_code
# 200 : 정상
# 404 : 없는 페이지

200

In [15]:
response.content # html의 바이너리 형식의 내용

b'<!DOCTYPE html>\r\n<html lang="en">\r\n<head>\r\n  <meta charset="UTF-8">\r\n</head>\r\n<body>\r\n  <h1 class="greeting css" id="text">Hello, CSS</h1>\r\n  <h1 class="css">Hi, CSS</h1>\r\n  <div id="subject">subject \xec\x84\xa0\xed\x83\x9d\xec\x9e\x90 \xec\x95\x88\xec\x9d\x98 \xeb\x82\xb4\xec\x9a\xa9</div>\r\n  <p>CSS \xec\x84\xa0\xed\x83\x9d\xec\x9e\x90\xeb\x8a\x94 \xeb\x8b\xa4\xec\x96\x91\xed\x95\x9c \xea\xb3\xb3\xec\x97\x90\xec\x84\x9c \xed\x99\x9c\xec\x9a\xa9\xeb\x90\xa9\xeb\x8b\x88\xeb\x8b\xa4</p>\r\n  <div class="contents">\r\n    \xec\x84\xa0\xed\x83\x9d\xec\x9e\x90\xeb\xa5\xbc \xec\x96\xb4\xeb\x96\xbb\xea\xb2\x8c \xec\x9e\x91\xec\x84\xb1\xed\x95\x98\xeb\x8a\x90\xeb\x83\x90\xec\x97\x90 \xeb\x94\xb0\xeb\x9d\xbc\r\n    <span>\xeb\x8b\xa4\xeb\xa5\xb8<b>\xec\x9a\x94\xec\x86\x8c\xea\xb0\x80 \xeb\xb0\x98\xed\x99\x98</b></span>\xeb\x90\xa9\xeb\x8b\x88\xeb\x8b\xa4\r\n  </div>\r\n  <div>CSS \xec\x84\xa0\xed\x83\x9d\xec\x9e\x90\xeb\x8a\x94 \xeb\x8b\xa4\xec\x96\x91\xed\x95\x9c \xea\xb3\

In [16]:
print(response.content.decode('utf-8'))

<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
</head>
<body>
  <h1 class="greeting css" id="text">Hello, CSS</h1>
  <h1 class="css">Hi, CSS</h1>
  <div id="subject">subject 선택자 안의 내용</div>
  <p>CSS 선택자는 다양한 곳에서 활용됩니다</p>
  <div class="contents">
    선택자를 어떻게 작성하느냐에 따라
    <span>다른<b>요소가 반환</b></span>됩니다
  </div>
  <div>CSS 선택자는 다양한 곳에 <b>활용</b>됩니다</div>
</body>
</html>


In [19]:
response.text

'<!DOCTYPE html>\r\n<html lang="en">\r\n<head>\r\n  <meta charset="UTF-8">\r\n</head>\r\n<body>\r\n  <h1 class="greeting css" id="text">Hello, CSS</h1>\r\n  <h1 class="css">Hi, CSS</h1>\r\n  <div id="subject">subject 선택자 안의 내용</div>\r\n  <p>CSS 선택자는 다양한 곳에서 활용됩니다</p>\r\n  <div class="contents">\r\n    선택자를 어떻게 작성하느냐에 따라\r\n    <span>다른<b>요소가 반환</b></span>됩니다\r\n  </div>\r\n  <div>CSS 선택자는 다양한 곳에 <b>활용</b>됩니다</div>\r\n</body>\r\n</html>'

In [24]:
# html 파싱 객체
from bs4 import BeautifulSoup
soup = BeautifulSoup(response.text, #response.content, 
                    "html.parser")
# soup

In [37]:
# 1. soup.select_one('선택자') : 해당 선택자 처음 하나 엘리먼트만 
el = soup.select_one('h1.css')
print('el =>', el)
print('el.text   =>', el.text)
print('el.string =>', el.string)
print('el의 속성들 =>', el.attrs)
print('el의 class속성 =>', el.attrs['class'])
print('el의 class속성 =>', el.attrs.get('class'))
# print('el의 href속성(없는 속성은 에러) =>', el.attrs.get['href'])
print('el의 href속성 =>', el.attrs.get('href'))
print('el의 name =>', el.name)

el => <h1 class="greeting css" id="text">Hello, CSS</h1>
el.text   => Hello, CSS
el.string => Hello, CSS
el의 속성들 => {'class': ['greeting', 'css'], 'id': 'text'}
el의 class속성 => ['greeting', 'css']
el의 class속성 => ['greeting', 'css']
el의 href속성 => None
el의 name => h1


In [45]:
# 2. soup.select('선택자') : 해당 선택자 엘리먼트 다 list로
els = soup.select('h1.css')
print('els =>', els)
print('els들의 text =>', [el.text for el in els])
print('els들의 string =>', [el.string for el in els])
print('els들의 속성들 =>', [el.attrs for el in els])
print('els들의 class 속성 =>', [el.attrs.get('class') for el in els])

els => [<h1 class="greeting css" id="text">Hello, CSS</h1>, <h1 class="css">Hi, CSS</h1>]
els들의 text => ['Hello, CSS', 'Hi, CSS']
els들의 string => ['Hello, CSS', 'Hi, CSS']
els들의 속성들 => [{'class': ['greeting', 'css'], 'id': 'text'}, {'class': ['css']}]
els들의 class 속성 => [['greeting', 'css'], ['css']]


In [50]:
# 3. soup.find(태그, 속성)  vs soup.select_one('선택자') : 해당 속성을 갖은 태그 처음 하나만 
print('select_one :', soup.select_one('h1.css'))
print('find       :', soup.find('h1', {'class':'css'}))
print('find       :', soup.find('h1', class_='css'))
print()
print('select_one :', soup.select_one('h1#text'))
print('select_one :', soup.find('h1', {'id':'text'}))

select_one : <h1 class="greeting css" id="text">Hello, CSS</h1>
find       : <h1 class="greeting css" id="text">Hello, CSS</h1>
find       : <h1 class="greeting css" id="text">Hello, CSS</h1>

select_one : <h1 class="greeting css" id="text">Hello, CSS</h1>
select_one : <h1 class="greeting css" id="text">Hello, CSS</h1>


In [61]:
# 4. soup.find_all(태그, 속성) vs. soup.select('선택자') : 해당 엘리먼트 다 list로
print('모든 h1.css와 span태그 :', soup.select('h1.css, span'))
print('모든 h1.css와 span태그 :', soup.find_all(['h1'], class_='css') +
                                soup.find_all('span'))

모든 h1.css와 span태그 : [<h1 class="greeting css" id="text">Hello, CSS</h1>, <h1 class="css">Hi, CSS</h1>, <span>다른<b>요소가 반환</b></span>]
모든 h1.css와 span태그 : [<h1 class="greeting css" id="text">Hello, CSS</h1>, <h1 class="css">Hi, CSS</h1>, <span>다른<b>요소가 반환</b></span>]


In [71]:
# 없는 엘리먼트 찾기
print('find_all(빈list) :', soup.find_all('a'))
print('find(None)       :', soup.find('a'))
print('select(빈list) :', soup.select('a'))
print('select_one(None) :', soup.select_one('a'))

find_all(빈list) : []
find(None)       : None
select(빈list) : []
select_one(None) : None


# 2절. 정적 웹 데이터 수집(정적 웹크롤링)
## 2.1 BeautifulSoup 모듈을 활용한 html 웹 데이터 수집
### 1) 환율정보 가져오기(네이버증권 > 시장지표)

- https://finance.naver.com/marketindex/

    * 크롤링 허용범위는 사이트마다 ~/robots.txt에서 확인할 수 있음
        - Allow : 크롤링 허용가능한 폴더
        - Disallow : 크롤링 제한 폴더

In [78]:
# soup객체 생성 방법1
import requests
from bs4 import BeautifulSoup
url = 'https://finance.naver.com/marketindex/'
response = requests.get(url)
# response # Response 
print(response.status_code)
# response.text # response.content
soup = BeautifulSoup(response.text, 'html.parser')

200


In [86]:
# soup객체 생성 방법2
from urllib.request import urlopen
url = 'https://finance.naver.com/marketindex/'
response = urlopen(url)
# response # HTTPResponse
print(response.status)
# print(response.read().decode('cp949'))
soup = BeautifulSoup(response, 'html.parser')

200


In [100]:
p = '1,417,000.70'
float(p.replace(',','')) # 방법1
float(''.join(p.split(','))) # 방법2

1417000.7

In [104]:
# div.head_info 밑의 span.value (find계열)
prices = []
headinfos = soup.find_all('div', class_='head_info')
for headinfo in headinfos:
    # print(headinfo)
    price = headinfo.find('span', class_='value')
    prices.append(float(''.join(price.text.split(','))))
print(prices)

[1417.7, 889.17, 1635.6, 210.09, 159.24, 1.1541, 1.3507, 99.71, 83.2, 1863.92, 4441.1, 200855.05]


In [109]:
# span.value (find계열)
price_els = soup.find_all('span', class_='value')
prices = [round(float(price.text.replace(',','')), 1) for price in price_els]
print(prices)

[1417.7, 889.2, 1635.6, 210.1, 159.2, 1.2, 1.4, 99.7, 83.2, 1863.9, 4441.1, 200855.0]


In [117]:
# 금액들 : div.head_info 밑의 span.value 
price_els = soup.select('div.head_info > span.value')
len(price_els)

12

In [116]:
# 타이틀
title_els = soup.select('h3.h_lst > span.blind')
len(title_els)

12

In [121]:
# 단위들 : div.head_info > span > span.blind
unit_els = soup.select('div.head_info > span > span.blind')
len(unit_els)
units = [unit_el.string for unit_el in unit_els]
units.insert(7, '')
units

['원', '원', '원', '원', '엔', '달러', '달러', '', '달러', '원', '달러', '원']

In [124]:
# 상승/하락 :  div.head_info > span.blind
trend_els = soup.select('div.head_info > span.blind')
#trend_els

In [125]:
len(title_els), len(price_els), len(units), len(trend_els)

(12, 12, 12, 12)

In [127]:
for idx in range(len(title_els)):
    print("{} : {} {} - {}".format(title_els[idx].text,
                                  price_els[idx].text,
                                  units[idx],
                                  trend_els[idx].text))

미국 USD : 1,417.70 원 - 상승
일본 JPY(100엔) : 889.17 원 - 상승
유럽연합 EUR : 1,635.60 원 - 상승
중국 CNY : 210.09 원 - 상승
달러/일본 엔 : 159.2400 엔 - 상승
유로/달러 : 1.1541 달러 - 하락
영국 파운드/달러 : 1.3507 달러 - 하락
달러인덱스 : 99.7100  - 상승
WTI : 83.2 달러 - 상승
휘발유 : 1863.92 원 - 하락
국제 금 : 4441.1 달러 - 상승
국내 금 : 200855.05 원 - 상승


In [128]:
for title, price, unit, trend in zip(title_els, price_els, units, trend_els):
    print("{} : {}{} - {}".format(title.text, price.text, unit, trend.text))

미국 USD : 1,417.70원 - 상승
일본 JPY(100엔) : 889.17원 - 상승
유럽연합 EUR : 1,635.60원 - 상승
중국 CNY : 210.09원 - 상승
달러/일본 엔 : 159.2400엔 - 상승
유로/달러 : 1.1541달러 - 하락
영국 파운드/달러 : 1.3507달러 - 하락
달러인덱스 : 99.7100 - 상승
WTI : 83.2달러 - 상승
휘발유 : 1863.92원 - 하락
국제 금 : 4441.1달러 - 상승
국내 금 : 200855.05원 - 상승


In [132]:
import pandas as pd
data = []
for title, price, unit, trend in zip(title_els, price_els, units, trend_els):
    data.append({'title' : title.text,
                 'price' : float(price.text.replace(',','')),
                 'unit' : unit,
                 'trend' : trend.text})
pd.DataFrame(data)#.to_csv('data/file.csv', index=False)

,title,price,unit,trend
0,미국 USD,1417.7000,원,상승
1,일본 JPY(100엔),889.1700,원,상승
2,유럽연합 EUR,1635.6000,원,상승
3,중국 CNY,210.0900,원,상승
4,달러/일본 엔,159.2400,엔,상승
5,유로/달러,1.1541,달러,하락
6,영국 파운드/달러,1.3507,달러,하락
7,달러인덱스,99.7100,,상승
8,WTI,83.2000,달러,상승
9,휘발유,1863.9200,원,하락


In [135]:
data = []
for title, price, unit, trend in zip(title_els, price_els, units, trend_els):
    data.append([title.text, float(price.text.replace(',','')), unit, trend.text])
pd.DataFrame(data, columns=['title','price','unit','trend'])

,title,price,unit,trend
0,미국 USD,1417.7000,원,상승
1,일본 JPY(100엔),889.1700,원,상승
2,유럽연합 EUR,1635.6000,원,상승
3,중국 CNY,210.0900,원,상승
4,달러/일본 엔,159.2400,엔,상승
5,유로/달러,1.1541,달러,하락
6,영국 파운드/달러,1.3507,달러,하락
7,달러인덱스,99.7100,,상승
8,WTI,83.2000,달러,상승
9,휘발유,1863.9200,원,하락


### 2) 이번주 로또번호 출력
- 방법2에서 User-Agent를 추가하여 soup생성
- https://search.daum.net/search?nil_suggest=btn&w=tot&DA=SBC&q=lotto (다음에서 lotto검색)
```
    1236회(2026.08.08 추첨)
    당첨번호 [12, 18, 21, 29, 34, 38]
    보너스 10
```

In [146]:
# 방법1
import requests
from bs4 import BeautifulSoup
url = 'https://search.daum.net/search?nil_suggest=btn&w=tot&DA=SBC&q=lotto'
response = requests.get(url)
print('response의 상태 :',response.status_code)
soup = BeautifulSoup(response.text, 'html.parser')
# soup

response의 상태 : 200


In [151]:
# 방법2
from urllib.request import urlopen, Request
headers = {'User-Agent':
          'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/151.0.0.0 Safari/537.36'}
request = Request(url, headers=headers)

response = urlopen(request)
print('response의 상태 :', response.status)
soup = BeautifulSoup(response, 'html.parser')
#soup

response의 상태 : 200


In [176]:
# 1236회 (2026.08.08 추첨)
# 당첨번호 [12, 18, 21, 29, 34, 38]
# 보너스 10
times = soup.select_one('div.prize span.f_red').text
date = soup.select_one('div.prize > span.date').text
title1 = soup.select_one('div.prize > strong').text[-4:]
lotto_numbers = soup.select('div.lottonum > span.ball:nth-last-child(n+3)')
lotto_numbers = soup.select('div.lottonum > span.ball')[:-2]
title2 = soup.select_one('div.lottonum span.screen_out').text
bonus_number = soup.select_one('div.lottonum > span.bg_ball1').text
print(times, date)
print(title1, [int(numbers.text) for numbers in lotto_numbers])
print(title2, bonus_number)

1236회 (2026.08.08 추첨)
당첨번호 [12, 18, 21, 29, 34, 38]
보너스 10


In [193]:
# 위의 select계열 함수를 find계열함수로 변경하여 구현해 보기
# times = soup.select_one('div.prize span.f_red').text
prize = soup.find('div', class_='prize')
times = prize.find('span', class_='f_red').text

# date = soup.select_one('div.prize > span.date').text
date = prize.find('span', class_='date').text

# title1 = soup.select_one('div.prize > strong').text[-4:]
title1 = prize.find('strong').text[-4:]

# lotto_numbers = soup.select('div.lottonum > span.ball')[:-2]
lottonum = soup.find('div', class_='lottonum')
lotto_numbers = lottonum.find_all('span', class_='ball')[:-2]

# title2 = soup.select_one('div.lottonum span.screen_out').text
title2 = lottonum.find('span', class_='screen_out').text

# bonus_number = soup.select_one('div.lottonum > span.bg_ball1').text
bonus_number = lottonum.find('span', class_='bg_ball1').text

print(times, date)
print(title1, [int(numbers.text) for numbers in lotto_numbers])
print(title2, bonus_number)

1236회 (2026.08.08 추첨)
당첨번호 [12, 18, 21, 29, 34, 38]
보너스 10


### 3) 다음 뉴스 검색 리스트
```
no title   href
0  타이틀1  http://~
1  타이틀2  http://~
2  타이틀3  http://~
```

In [2]:
# 방법1
import requests
from bs4 import BeautifulSoup
word = '개미들'
url = f'https://search.daum.net/search?w=news&q={word}&enc=utf8&cluster=y&cluster_page=1&DA=DNS'
print(url)
response = requests.get(url)
print(response.status_code)
soup = BeautifulSoup(response.text, 'html.parser')

https://search.daum.net/search?w=news&q=개미들&enc=utf8&cluster=y&cluster_page=1&DA=DNS
200


In [213]:
# 방법2
from urllib.request import urlopen, Request
from urllib.parse import quote
word = quote('비트코인')
url = f'https://search.daum.net/search?w=news&q={word}&enc=utf8&cluster=y&cluster_page=1&DA=DNS'
print(url)
headers = {'User-Agent':
          'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/151.0.0.0 Safari/537.36'}
# request = Request(url, headers=headers)
request = Request(url)
request.add_header('User-Agent', 
                  'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/151.0.0.0 Safari/537.36')
response = urlopen(request)
print(response.status)
soup = BeautifulSoup(response, 'html.parser')
# soup

https://search.daum.net/search?w=news&q=%EB%B9%84%ED%8A%B8%EC%BD%94%EC%9D%B8&p=1
200


In [3]:
items_find_list = [] # 검색한 결과를 담을 dict 리스트
items_el = soup.select('div.item-title > strong.tit-g > a')
for idx, item in enumerate(items_el):
    # print(idx, item)
    items_find_list.append({'no':idx,
                            'title':item.text,
                            'link':item.attrs.get('href')})
import pandas as pd
pd.DataFrame(items_find_list)

,no,title,link
0,0,"""불기둥 무서워""…'천스닥' 전망에도 개미들 '하락 베팅'",http://v.daum.net/v/20260811163537806
1,1,서학개미들 'SK하닉' 아닌 여기 몰렸다,http://v.daum.net/v/20260808095848252
2,2,"“주식 잃고 대출도 막혔다”…600조 날린 개미들, 삼전닉스에 다시 희망을?[주형...",http://v.daum.net/v/20260808055128543
3,3,국장 뜨는 개미들과 딴판…‘큰 손’ 블랙록이 담은 종목은,http://v.daum.net/v/20260812093738363
4,4,장기투자 하라더니 정책은 오락가락…개미들 다시 美 증시로,http://v.daum.net/v/20260810151107970
5,5,"“고점 물린 레버리지 ETF 개미들, 本株·일반 ETF로 갈아타라”",http://v.daum.net/v/20260811003738083
6,6,'200조 쏜대' 개미들 환호…일본서 또 '사상 최고' 배당 뜬다,http://v.daum.net/v/20260813073709577
7,7,"개미들 이달 '코스피 베팅', 외국인과 정반대 흐름",http://v.daum.net/v/20260813082250549
8,8,개미들 증시에 질렸다?…투자 예탁금 100조 붕괴 코앞,http://v.daum.net/v/20260805114326291
9,9,"""내 주식 휴지조각 되나""…상폐 공포에 떠는 개미들 '발 동동'",http://v.daum.net/v/20260813075209804


In [227]:
items_find_list = [] # 검색한 결과를 담을 2차원 리스트
items_el = soup.select('div.item-title > strong.tit-g > a')
for idx, item in enumerate(items_el):
    items_find_list.append([idx, item.text, item.attrs.get('href')])
pd.DataFrame(items_find_list, columns=['순번','기사제목','링크'])

,순번,기사제목,링크
0,0,9000만원 깨진 비트코인…美물가지수 앞두고 위험회피 심리↑,http://v.daum.net/v/20260812104759348
1,1,[코인뉴스] 비트코인 박스권 지속…다음 변수는?,http://v.daum.net/v/20260812093050056
2,2,[코인뉴스] CPI 앞둔 비트코인…박스권 속 엇갈린 베팅,http://v.daum.net/v/20260812163039451
3,3,9000만원대 갇힌 비트코인…'100만弗 vs 4만弗' 극단 전망,http://v.daum.net/v/20260812161211645
4,4,[코인시세] 비트코인 6만3천달러대 약세…CPI 경계감 지속,http://v.daum.net/v/20260812103841887
5,5,"[08:03 가상자산] 비트코인, 美 CPI 결과 발표 앞두고 9000만원 선 붕괴",http://v.daum.net/v/20260812080815138
6,6,"트럼프 회사, 비트코인에 3300억 베팅했다 '쓴맛'⋯손실 12배 늘었다",http://v.daum.net/v/20260811144238992
7,7,"금리 공포 걷히자 ""금값 조정 끝났다""… 비트코인은 나 홀로 약세",http://v.daum.net/v/20260812164709172
8,8,[아주경제 코이너스 브리핑] 호르무즈 불확실성에…비트코인 6만3000달러대 횡보,http://v.daum.net/v/20260812082712570
9,9,비트코인 다시 6.3만달러대로…美 CPI 발표 ‘촉각’ [코인 모닝콜],http://v.daum.net/v/20260812082137419


In [4]:
# 다음 뉴스 검색 함수(원하는 키워드, 원하는 페이지로)
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
def collect_list(keyword, page):
    'keyword로 해당 page에 검색한 결과 dict list return'
    # url = f'https://search.daum.net/search?w=news&q={keyword}&enc=utf8&cluster=y&cluster_page=1&DA=PGD&p={page}'
    url = 'https://search.daum.net/search?w=news&enc=utf8&cluster=y&cluster_page=1&DA=PGD'
    params = {'q':keyword, 'p':page}
    response = requests.get(url, params=params)
    soup = BeautifulSoup(response.text, 'html.parser')
    items_find_list = []
    items_el = soup.select('div.item-title > strong.tit-g > a')
    for idx, item in enumerate(items_el):
        items_find_list.append({'no':(page-1)*10 + idx, 
                                'title':item.text,
                                'link':item.attrs.get('href')})
    return items_find_list

In [5]:
collect_list('태풍',2)

[{'no': 10,
  'title': ' 블랙핑크 지수 ‘청바지에 흰티면 OK!’[포토엔HD] ',
  'link': 'http://v.daum.net/v/20260811003242018'},
 {'no': 11,
  'title': ' 송혜교, 흰티에 청바지 일상 사진 공개하자..절친 다 몰려왔다 ',
  'link': 'http://v.daum.net/v/20260806151007109'},
 {'no': 12,
  'title': ' 흰 티에 청바지…송혜교, 폭염 속에도 빛나는 비주얼 [N샷] ',
  'link': 'http://v.daum.net/v/20260806151445411'},
 {'no': 13,
  'title': ' ‘똥배 논란’ 사라지더니…홍진영, 오조오억년 만에 치마 벗고 청바지 ',
  'link': 'http://v.daum.net/v/20260722074501440'},
 {'no': 14,
  'title': ' 올가을 청바지는 더 이상 기본템이 아닙니다 ',
  'link': 'http://v.daum.net/v/20260810234410440'},
 {'no': 15,
  'title': ' "지퍼 올려!" 엄마들 질색한 청바지…Z세대는 열광한다는데 ',
  'link': 'http://v.daum.net/v/20260724132009295'},
 {'no': 16,
  'title': ' 여름에도 교복처럼 입을 청바지 발견했습니다 ',
  'link': 'http://v.daum.net/v/20260722183437854'},
 {'no': 17,
  'title': ' 돌싱 박지윤, 8㎏ 빼고 또 다이어트 하더니 청바지핏 완벽 ',
  'link': 'http://v.daum.net/v/20260724152711935'},
 {'no': 18,
  'title': ' 여름과 가장 잘 어울리는 청바지는 따로 있습니다 ',
  'link': 'http://v.daum.net/v/20260724180507627'},
 {

In [11]:
r = []
r.extend([1, 2, 3])
r.extend([4, 5, 6])
r

[1, 2, 3, 4, 5, 6]

In [12]:
result = [] # 해당 키워드 원하는 페이지 수만큼 검색한 결과를 담을 변수 dict 리스트
pages = 3
for page in range(1, pages+1):
    print(f'== {page} 페이지 수집 중 ==')
    item_result = collect_list('추석열차', page)
    result.extend(item_result)
    time.sleep(3)
pd.DataFrame(result)

== 1 페이지 수집 중 ==
== 2 페이지 수집 중 ==
== 3 페이지 수집 중 ==


,no,title,link
0,0,"추석 열차표 예매, KTX·SRT 따로 안 해도 된다…9월 1일 통합 앱 출시",http://v.daum.net/v/20260802115907094
1,1,“올해 추석부터 열차값 10% 아끼세요”…KTX·SRT 9월 통합,http://v.daum.net/v/20260802192100597
2,2,"나중엔 늦는다…추석·10월에 3일 연차로 9일 황금연휴 완성, 어디에 걸까? [여...",http://v.daum.net/v/20260807123245162
3,3,'올해도 포기했는데' 설렌다…추석 앞두고 벌어진 대반전,http://v.daum.net/v/20260812190246138
4,4,"""지금 안 사면 늦는다""…직장인들 난리 난 황금연휴, 항공권·숙소 예약전쟁",http://v.daum.net/v/20260812105728812
5,5,"[현장]코레일·SR 통합…추석, 통합운영 시험대",http://v.daum.net/v/20260803145005360
6,6,"'대국민 티켓팅' 경쟁률 낮아질까…'최대 17,000석' 는다",http://v.daum.net/v/20260813071804287
7,7,KTX·SRT 9월부터 통합…부산·울산 추석 귀성표 전쟁 숨통 트이나,http://v.daum.net/v/20260805123449011
8,8,"코레일-SRT, 철도 회원 통합 시작…""9월 운행 열차부터 통합 예매""",http://v.daum.net/v/20260713171250054
9,9,"철도, 좌석 주간 11만 석 확대 및 예매 앱 단일화",http://v.daum.net/v/20260812202647188


In [16]:
keywords = ['김혜수', '영화']
pages = 3
result0 = [] # keyword[0] 1~pages페이지까지 검색한 결과 dict list
result1 = [] # keyword[1] 1~pages페이지까지 검색한 결과 dict list
for i, keyword in enumerate(keywords):
    print(f'= = {i+1}번째 검색어 {keyword} 검색 결과 수집({pages}페이지) 중입니다 = =')
    for page in range(1, pages+1):
        if i==0:
            result0.extend(collect_list(keyword, page))
        else:
            result1.extend(collect_list(keyword, page))
        time.sleep(3)

= = 1번째 검색어 김혜수 검색 결과 수집(3페이지) 중입니다 = =
= = 2번째 검색어 영화 검색 결과 수집(3페이지) 중입니다 = =


In [17]:
result0_df = pd.DataFrame(result0)
result1_df = pd.DataFrame(result1)
result0_df.sample()

,no,title,link
3,3,‘지금 불륜’ 김혜수 55세 맞아? 감탄 부르는 비주얼,http://v.daum.net/v/20260803172404089


In [18]:
result1_df.head()

,no,title,link
0,0,여름 대작 피했더니 9월에 6편 몰린 한국영화… ‘공멸’ 잔혹사 끊을까 [영화 뷰],http://v.daum.net/v/20260812082840601
1,1,"[단독] 염정아, 스릴러 영화 '투피스'서 문가영과 만난다! 꿈의 조합 완성",http://v.daum.net/v/20260813065649908
2,2,"'오디세이' 봤다면, 이 영화도 함께 보세요",http://v.daum.net/v/20260811160736520
3,3,"다대포 바다, 영화로 물든다…제4회 다대포선셋영화축제 14일 개막",http://v.daum.net/v/20260813101444334
4,4,"“우리 증조부도 매국노, 조상 업보 청산중”…영화 ‘암살’ 이경영 후손의 고백",http://v.daum.net/v/20260813094909942


In [20]:
result0_df.to_csv(f'data/ch14_{keywords[0]}.csv', index=False, encoding='cp949')
result1_df.to_csv(f'data/ch14_{keywords[1]}.csv', index=False, encoding='cp949')

### 4) User-Agent를 추가하여 크롤링
- request.get(url), urlopen(url)함수를 사용하면 크롤링이 막혀있는 사이트
- 방법2에서 User-Agent를 추가혀여 크롤링

- https://www.melon.com/robots.txt 에서 일부 경로는 User-Agent에 봇이 지정

In [24]:
# 방법1
import requests
from bs4 import BeautifulSoup
url = 'https://www.melon.com/chart/'
melonResponse = requests.get(url)
print(melonResponse.status_code)
soup = BeautifulSoup(melonResponse.text, 'html.parser')
soup

406


In [26]:
# 방법2
from urllib.request import urlopen
from bs4 import BeautifulSoup
url = 'https://www.melon.com/chart/'
#melonResponse = urlopen(url) # HTTPError: HTTP Error 406: Not Acceptable

In [30]:
# User-Agent를 추가하여 방법2
from urllib.request import urlopen, Request
from bs4 import BeautifulSoup
url = 'https://www.melon.com/chart/'
headers = {'user-agent':
          'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/151.0.0.0 Safari/537.36'}
# melonpage = Request(url, headers=headers)
melonpage = Request(url)
melonpage.add_header('user-agent',
                    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/151.0.0.0 Safari/537.36')
melonResponse = urlopen(melonpage)
print(melonResponse.status)
soup = BeautifulSoup(melonResponse, 'html.parser')
# soup

200


In [35]:
# User-Agent를 추가하여 방법1
import requests
from bs4 import BeautifulSoup
url = 'https://www.melon.com/chart/'
headers = {'user-agent':
          'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/151.0.0.0 Safari/537.36'}
melonResponse = requests.get(url, headers=headers)
print(melonResponse.status_code)
soup = BeautifulSoup(melonResponse.text, 'html.parser')
#soup

200


In [76]:
# 1위 : LOVE ATTACK | RESCENE (리센느)의 url {}
# 순위, 곡명, 가수, 가수페이지
rank_els = soup.select('div.wrap.t_center > span.rank')[1:] # 맨앞엔 '순위'
ranks = [rank.text for rank in rank_els]

title_els = soup.select('div.ellipsis.rank01 > span > a')
titles = [title.text for title in title_els]

singer_els = soup.select('span.checkEllipsis') # 40위, 90위 가수가 복수명
singers = [singer.text.replace('\xa0','') for singer in singer_els]

links = []
for singer_el in singer_els:
    # print(singer_el)
    singer_link = 'https://www.melon.com' + singer_el.find('a').attrs.get('href')
    links.append(singer_link)
len(ranks), len(titles), len(singers), len(links)

# for idx, (title, singer, link) in enumerate(zip(titles, singers, links)):
#     print("{}위. {} | {}".format(idx+1, title, singer))
melon_chat_list = []
for rank, title, singer, link in zip(ranks, titles, singers, links):
    # print(f'{rank}위 {title} | {singer}')
    melon_chat_list.append({
        '순위':rank,
        '곡명':title,
        '가수':singer,
        '가수페이지':link
    })
pd.DataFrame(melon_chat_list) # pd.options.display.max_rows(60)행이상은 중간이 생략

,순위,곡명,가수,가수페이지
0,1,LOVE ATTACK,RESCENE (리센느),https://www.melon.com/artist/detail.htm?artist...
1,2,갑자기,아이오아이 (I.O.I),https://www.melon.com/artist/detail.htm?artist...
2,3,REDRED,CORTIS (코르티스),https://www.melon.com/artist/detail.htm?artist...
3,4,LEMONADE,aespa,https://www.melon.com/artist/detail.htm?artist...
4,5,Pretty Girl,RESCENE (리센느),https://www.melon.com/artist/detail.htm?artist...
...,...,...,...,...
95,96,BLACKHOLE,IVE (아이브),https://www.melon.com/artist/detail.htm?artist...
96,97,FOCUS,Hearts2Hearts (하츠투하츠),https://www.melon.com/artist/detail.htm?artist...
97,98,Soda Pop,"KPop Demon Hunters Cast, Danny Chung, Saja Boy...",https://www.melon.com/artist/detail.htm?artist...
98,99,OVERDRIVE,TWS (투어스),https://www.melon.com/artist/detail.htm?artist...


### 5) 네이버 지식인 검색(open API 사용X)
- 특정 keyword를 특정 페이지 수만큼

In [78]:
# 방법1
from requests import get
from bs4 import BeautifulSoup
keyword = '쳇지피티'
url = f'https://kin.naver.com/search/list.naver?query={keyword}'
print(url)
response = get(url)
print(response.status_code)
soup = BeautifulSoup(response.text, 'html.parser')

https://kin.naver.com/search/list.naver?query=쳇지피티
200


In [83]:
# 방법2
from urllib.request import urlopen
from bs4 import BeautifulSoup
from urllib.parse import quote
keyword = quote('쳇지피티')
url = f'https://kin.naver.com/search/list.naver?query={keyword}'
print(url)
response = urlopen(url)
print(response.status)
soup = BeautifulSoup(response, 'html.parser')

https://kin.naver.com/search/list.naver?query=%EC%B3%87%EC%A7%80%ED%94%BC%ED%8B%B0
200


In [87]:
# keyword를 원하는 페이지 수만큼
# 방법1
from requests import get
from bs4 import BeautifulSoup
keyword = '쳇지피티'
pages = 2
items_list = [] # 크롤링한 데이터를 담을 list
for page in range(1, pages+1):
    url = 'https://kin.naver.com/search/list.naver?query={keyword}&page={page}'

1
2
